# Lecture 12 · Attention by hand · on a toy alignment task

**Goal** · build scaled dot-product attention from scratch, visualize the attention heatmap, see it solve a task an RNN would struggle with.

Task · *copy* — given a random sequence of 8 tokens, learn to produce the same sequence in reverse. Requires pointing at the right input token at every output step.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math

torch.manual_seed(0)

## 1 · Scaled dot-product attention · 5 lines

In [ ]:
def attention(Q, K, V):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

## 2 · Tiny worked example by hand

Three keys, three values, one query. Watch the blending happen.

In [ ]:
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.2, 0.1]])   # 3 keys
V = torch.tensor([[10.0, 0.0], [0.0, 10.0], [1.0, 1.0]])   # 3 values
Q = torch.tensor([[0.1, 0.9]])                              # 1 query

out, w = attention(Q, K, V)
print(f'attention weights · {w[0].numpy()}')
print(f'output · {out[0].numpy()}')
print()
print('The query [0.1, 0.9] matches key 2 (0, 1) most strongly.')
print('Most of the output comes from V[1] = [0, 10].')

## 3 · Toy seq2seq · reverse a random sequence with attention

Input · random 8-digit sequence. Output · same sequence reversed.

Without attention this needs perfect memorization. With attention we just teach the decoder to point.

In [ ]:
V_SIZE = 10
L = 8

def gen_batch(batch):
    src = torch.randint(0, V_SIZE, (batch, L))
    tgt = src.flip(-1)
    return src, tgt

src, tgt = gen_batch(3)
print('src', src[0].tolist())
print('tgt', tgt[0].tolist())

## 4 · Tiny encoder-decoder with attention

In [ ]:
class ReverseNet(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.emb = nn.Embedding(V_SIZE, d)
        self.pos = nn.Embedding(L, d)
        self.Wq = nn.Linear(d, d)
        self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d)
        self.out = nn.Linear(d, V_SIZE)

    def forward(self, src):
        B = src.size(0)
        pos = torch.arange(L, device=src.device)
        x = self.emb(src) + self.pos(pos)[None]
        # learn decoder queries directly from position
        decoder_pos = self.pos(pos)[None].expand(B, -1, -1)
        Q = self.Wq(decoder_pos)
        K = self.Wk(x)
        V = self.Wv(x)
        attn_out, weights = attention(Q, K, V)
        return self.out(attn_out), weights

In [ ]:
model = ReverseNet()
opt = torch.optim.Adam(model.parameters(), lr=5e-3)

for step in range(1000):
    src, tgt = gen_batch(64)
    logits, _ = model(src)
    loss = F.cross_entropy(logits.reshape(-1, V_SIZE), tgt.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        pred = logits.argmax(-1)
        acc = (pred == tgt).float().mean()
        print(f'step {step:4d}  loss {loss.item():.3f}  acc {acc.item():.2f}')

## 5 · Visualize the learned attention

If attention really learned to "reverse-point," the heatmap should show a strong anti-diagonal.

In [ ]:
src, tgt = gen_batch(1)
with torch.no_grad():
    logits, weights = model(src)
w = weights[0].numpy()

plt.figure(figsize=(5, 5))
plt.imshow(w, cmap='Oranges', aspect='auto')
plt.xlabel('source position')
plt.ylabel('target position')
plt.title(f'attention heatmap\nsrc={src[0].tolist()}')
for i in range(L):
    for j in range(L):
        plt.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(); plt.tight_layout(); plt.show()

print(f'pred · {logits.argmax(-1)[0].tolist()}')
print(f'tgt  · {tgt[0].tolist()}')

**Observe** · the attention weights concentrate on the **anti-diagonal** · target position $i$ attends to source position $L-1-i$. The network learned the reversal pattern without us programming it in.